In [2]:
import ee
import geemap

# ============================================================
# 1️⃣ INITIALIZE EARTH ENGINE
# ============================================================
ee.Authenticate()
ee.Initialize(project='gee-research123456')

# ============================================================
# 2️⃣ LOAD AOI
# ============================================================
tekeze_fc = ee.FeatureCollection(
    'projects/gee-research123456/assets/tekeze'
)
tekeze = tekeze_fc.geometry()

# ============================================================
# 3️⃣ SETTINGS
# ============================================================
startYear = 1981   # ⚠️ start small first
endYear = 1989     # ⚠️ test with few years
outFolder = 'CHIRPS_Monthly_tekeze'

# ============================================================
# 4️⃣ LOAD CHIRPS DATA
# ============================================================
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(tekeze))

# ============================================================
# 5️⃣ MONTHLY FUNCTION
# ============================================================
def monthlySum(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    return (chirps
            .filterDate(start, end)
            .sum()
            .clip(tekeze)
            .set({
                'system:time_start': start.millis()
            }))

# ============================================================
# 6️⃣ EXPORT LOOP (SAFE VERSION)
# ============================================================
for year in range(startYear, endYear + 1):
    for month in range(1, 13):

        img = monthlySum(year, month)

        date = f"{year}{month:02d}"

        task = ee.batch.Export.image.toDrive(
            image=img,
            description='CHIRPS_tekeze_' + date,
            folder=outFolder,
            fileNamePrefix='CHIRPS_tekeze_' + date,
            region=tekeze,
            scale=5000,
            crs='EPSG:4326',
            maxPixels=1e13
        )

        task.start()
        print("✅ Export started:", date)

✅ Export started: 198101
✅ Export started: 198102
✅ Export started: 198103
✅ Export started: 198104
✅ Export started: 198105
✅ Export started: 198106
✅ Export started: 198107
✅ Export started: 198108
✅ Export started: 198109
✅ Export started: 198110
✅ Export started: 198111
✅ Export started: 198112
✅ Export started: 198201
✅ Export started: 198202
✅ Export started: 198203
✅ Export started: 198204
✅ Export started: 198205
✅ Export started: 198206
✅ Export started: 198207
✅ Export started: 198208
✅ Export started: 198209
✅ Export started: 198210
✅ Export started: 198211
✅ Export started: 198212
✅ Export started: 198301
✅ Export started: 198302
✅ Export started: 198303
✅ Export started: 198304
✅ Export started: 198305
✅ Export started: 198306
✅ Export started: 198307
✅ Export started: 198308
✅ Export started: 198309
✅ Export started: 198310
✅ Export started: 198311
✅ Export started: 198312
✅ Export started: 198401
✅ Export started: 198402
✅ Export started: 198403
✅ Export started: 198404


### Visualization for 2000-2002

In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import math

# ============================================================
# 1️⃣ DATA FOLDER
# ============================================================
folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly"

# ============================================================
# 2️⃣ LIST FILES
# ============================================================
files = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])
files = [f for f in files if 200001 <= int(f.split("_")[-1].split(".")[0]) <= 200212]

n = len(files)

# ============================================================
# 3️⃣ GRID SETTINGS
# ============================================================
cols = 5
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(11, rows * 1.8))
axes = axes.flatten()

# ============================================================
# 4️⃣ LOOP & PLOT
# ============================================================
for i, file in enumerate(files):
    path = os.path.join(folder, file)
    
    date = file.split("_")[-1].split(".")[0]
    year = date[:4]
    month = date[4:]
    
    with rasterio.open(path) as src:
        data = src.read(1)
        extent = [src.bounds.left, src.bounds.right,
                  src.bounds.bottom, src.bounds.top]
    
    im = axes[i].imshow(
        data,
        extent=extent,
        cmap='YlGnBu',
        vmin=0,
        vmax=500
    )
    
    axes[i].set_title(f"{year}-{month}", fontsize=7)

    # Determine row & column
    row = i // cols
    col = i % cols

    # ✔ Show latitude ONLY on left column
    if col == 0:
        axes[i].set_ylabel("Lat", fontsize=7)
    else:
        axes[i].set_yticks([])

    # ✔ Show longitude ONLY on bottom row
    if row == rows - 1:
        axes[i].set_xlabel("Lon", fontsize=7)
    else:
        axes[i].set_xticks([])

# Hide empty plots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# ============================================================
# 5️⃣ COLORBAR (clean placement)
# ============================================================
cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Rainfall (mm/month)', fontsize=9)

# ============================================================
# 6️⃣ TIGHT SPACING (VERY IMPORTANT)
# ============================================================
plt.subplots_adjust(
    left=0.06,
    right=0.9,
    top=0.95,
    bottom=0.07,
    wspace=0.02,   # 🔥 reduced gap horizontally
    hspace=0.15    # 🔥 reduced gap vertically
)

plt.show()

### SPI

In [ ]:
import os
import numpy as np
import rasterio
from scipy.stats import gamma, norm

# ============================================================
# 1️⃣ LOAD DATA (ALL YEARS 1990–2025)
# ============================================================
folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly"

files = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])

data_list = []
for f in files:
    with rasterio.open(os.path.join(folder, f)) as src:
        data_list.append(src.read(1))
        meta = src.meta

data = np.stack(data_list)   # shape (time, rows, cols)
t, r, c = data.shape

print("Data shape:", data.shape)

# ============================================================
# 2️⃣ STEP A — MOVING SUM (x^(k))
# ============================================================
def moving_sum(ts, k):
    out = np.full_like(ts, np.nan, dtype=float)
    for i in range(k - 1, len(ts)):
        out[i] = np.sum(ts[i - k + 1:i + 1])
    return out

# ============================================================
# 3️⃣ STEP B–E — SPI FUNCTION (FULL EQUATION)
# ============================================================
def compute_spi(ts, scale):

    # Step A: aggregated rainfall
    xk = moving_sum(ts, scale)

    # Remove NaN
    valid = xk[~np.isnan(xk)]

    if len(valid) < 30:
        return np.full_like(ts, np.nan)

    # Step D: probability of zero rainfall
    q = np.sum(valid == 0) / len(valid)

    # Step B: fit gamma on NON-ZERO values
    non_zero = valid[valid > 0]

    if len(non_zero) < 10:
        return np.full_like(ts, np.nan)

    try:
        alpha, loc, beta = gamma.fit(non_zero, floc=0)
    except:
        return np.full_like(ts, np.nan)

    spi = np.full_like(ts, np.nan, dtype=float)

    # Step C–E: compute SPI
    for i in range(len(xk)):
        if not np.isnan(xk[i]):

            if xk[i] == 0:
                H = q
            else:
                G = gamma.cdf(xk[i], alpha, loc=loc, scale=beta)
                H = q + (1 - q) * G   # adjusted probability

            # avoid infinity
            H = np.clip(H, 1e-6, 1 - 1e-6)

            # Step E: standard normal transform
            spi[i] = norm.ppf(H)

    return spi

# ============================================================
# 4️⃣ COMPUTE SPI FOR ALL PIXELS
# ============================================================
SPI1 = np.full((t, r, c), np.nan)
SPI3 = np.full((t, r, c), np.nan)
SPI6 = np.full((t, r, c), np.nan)

for i in range(r):
    print(f"Processing row {i+1}/{r}")
    
    for j in range(c):
        ts = data[:, i, j]

        if np.all(np.isnan(ts)):
            continue

        SPI1[:, i, j] = compute_spi(ts, 1)
        SPI3[:, i, j] = compute_spi(ts, 3)
        SPI6[:, i, j] = compute_spi(ts, 6)

print("✅ SPI computed successfully")

# ============================================================
# 5️⃣ SAVE OUTPUT
# ============================================================
out_folder = os.path.join(folder, "SPI_outputs")
os.makedirs(out_folder, exist_ok=True)

meta.update(dtype=rasterio.float32)

def save_spi(spi_data, name):
    for k in range(t):
        out_path = os.path.join(out_folder, f"{name}_{k:03d}.tif")
        
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(spi_data[k].astype(np.float32), 1)

save_spi(SPI1, "SPI1")
save_spi(SPI3, "SPI3")
save_spi(SPI6, "SPI6")

print("✅ All SPI maps saved!")

### Renaming

In [ ]:
import os

# ============================================================
# 1️⃣ FOLDERS
# ============================================================
chirps_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly"
spi_folder = os.path.join(chirps_folder, "SPI_outputs")

# ============================================================
# 2️⃣ GET FILE LISTS
# ============================================================
chirps_files = sorted([f for f in os.listdir(chirps_folder) if f.endswith(".tif")])
spi_files = sorted([f for f in os.listdir(spi_folder) if f.endswith(".tif")])

print("CHIRPS files:", len(chirps_files))
print("SPI files:", len(spi_files))

# ============================================================
# 3️⃣ RENAME FUNCTION
# ============================================================
def rename_spi(prefix):
    # filter SPI files (SPI1, SPI3, SPI6)
    files = sorted([f for f in spi_files if f.startswith(prefix)])
    
    for i, f in enumerate(files):
        old_path = os.path.join(spi_folder, f)
        
        # extract date from CHIRPS
        date = chirps_files[i].split("_")[-1].split(".")[0]
        
        new_name = f"{prefix}_{date}.tif"
        new_path = os.path.join(spi_folder, new_name)
        
        os.rename(old_path, new_path)
        
    print(f"✅ Renamed {prefix}")

# ============================================================
# 4️⃣ APPLY
# ============================================================
rename_spi("SPI1")
rename_spi("SPI3")
rename_spi("SPI6")

print("🎉 All files renamed successfully!")

### timeseries plot, extraction and analysis

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# 1️⃣ FOLDERS
# ============================================================
spi_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly\SPI_outputs"

out_folder = os.path.join(spi_folder, "Analysis")
os.makedirs(out_folder, exist_ok=True)

# ============================================================
# 2️⃣ GET FILES
# ============================================================
def get_files(prefix):
    return sorted([f for f in os.listdir(spi_folder) if f.startswith(prefix)])

spi1_files = get_files("SPI1")
spi3_files = get_files("SPI3")
spi6_files = get_files("SPI6")

# ============================================================
# 3️⃣ VISUALIZATION FUNCTION
# ============================================================
def plot_maps(files, name):

    out_maps = os.path.join(out_folder, f"{name}_maps")
    os.makedirs(out_maps, exist_ok=True)

    for f in files:
        path = os.path.join(spi_folder, f)

        date = f.split("_")[-1].split(".")[0]

        with rasterio.open(path) as src:
            data = src.read(1)
            extent = [src.bounds.left, src.bounds.right,
                      src.bounds.bottom, src.bounds.top]

        plt.figure(figsize=(5,4))

        img = plt.imshow(
            data,
            extent=extent,
            cmap='RdBu',     # 🔥 best for SPI
            vmin=-2.5,
            vmax=2.5
        )

        plt.colorbar(img, label="SPI")
        plt.title(f"{name} - {date}")

        plt.xticks([])
        plt.yticks([])

        plt.tight_layout()

        out_path = os.path.join(out_maps, f"{name}_{date}.png")
        plt.savefig(out_path, dpi=150)
        plt.close()

    print(f"✅ Maps saved for {name}")

# ============================================================
# 4️⃣ TIME SERIES FUNCTION
# ============================================================
def extract_timeseries(files, name):

    dates = []
    values = []

    for f in files:
        path = os.path.join(spi_folder, f)

        date = f.split("_")[-1].split(".")[0]

        with rasterio.open(path) as src:
            data = src.read(1)

            # mean over basin
            mean_val = np.nanmean(data)

        dates.append(date)
        values.append(mean_val)

    df = pd.DataFrame({
        "Date": dates,
        name: values
    })

    # Save CSV
    csv_path = os.path.join(out_folder, f"{name}_timeseries.csv")
    df.to_csv(csv_path, index=False)

    # Plot
    plt.figure(figsize=(10,4))
    plt.plot(df["Date"], df[name], color='black')

    plt.axhline(0, linestyle='--')  # normal line

    plt.title(f"{name} Time Series")
    plt.xticks(rotation=90)

    plt.tight_layout()

    plot_path = os.path.join(out_folder, f"{name}_timeseries.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()

    print(f"✅ Time series saved for {name}")

# ============================================================
# 5️⃣ RUN EVERYTHING
# ============================================================
plot_maps(spi1_files, "SPI1")
plot_maps(spi3_files, "SPI3")
plot_maps(spi6_files, "SPI6")

extract_timeseries(spi1_files, "SPI1")
extract_timeseries(spi3_files, "SPI3")
extract_timeseries(spi6_files, "SPI6")

print("🎉 ALL DONE!")

### CREATE DROUGHT MAPS

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

spi_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly\SPI_outputs"
out_folder = os.path.join(spi_folder, "Drought_Maps")
os.makedirs(out_folder, exist_ok=True)

# ============================================================
# CLASSIFICATION FUNCTION
# ============================================================
def classify_spi(data):

    out = np.full_like(data, np.nan)

    out[(data >= 2)] = 3
    out[(data >= 1.5) & (data < 2)] = 2
    out[(data >= 1) & (data < 1.5)] = 1
    out[(data > -1) & (data < 1)] = 0
    out[(data <= -1) & (data > -1.5)] = -1
    out[(data <= -1.5) & (data > -2)] = -2
    out[(data <= -2)] = -3

    return out

# ============================================================
# PROCESS SPI1 (you can repeat for SPI3, SPI6)
# ============================================================
files = sorted([f for f in os.listdir(spi_folder) if f.startswith("SPI3")])

for f in files:
    path = os.path.join(spi_folder, f)

    date = f.split("_")[-1].split(".")[0]

    with rasterio.open(path) as src:
        data = src.read(1)
        meta = src.meta
        extent = [src.bounds.left, src.bounds.right,
                  src.bounds.bottom, src.bounds.top]

    classified = classify_spi(data)

    # Save raster
    meta.update(dtype=rasterio.int16)
    out_tif = os.path.join(out_folder, f"Drought_{date}.tif")

    with rasterio.open(out_tif, 'w', **meta) as dst:
        dst.write(classified.astype(np.int16), 1)

    # Plot map
    plt.figure(figsize=(5,4))

    cmap = plt.get_cmap("RdBu", 7)

    plt.imshow(classified, cmap=cmap, vmin=-3, vmax=3)
    plt.title(f"Drought Map {date}")

    plt.colorbar(label="Drought Class")

    plt.xticks([])
    plt.yticks([])

    plt.tight_layout()

    out_png = os.path.join(out_folder, f"Drought_{date}.png")
    plt.savefig(out_png, dpi=150)
    plt.close()

print("✅ Drought maps created!")

### DETECT DROUGHT PERIODS

In [ ]:
import pandas as pd
import os

analysis_folder = os.path.join(
    r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly\SPI_outputs",
    "Analysis"
)

df = pd.read_csv(os.path.join(analysis_folder, "SPI3_timeseries.csv"))

droughts = []
in_drought = False

for i in range(len(df)):
    val = df["SPI3"].iloc[i]
    date = df["Date"].iloc[i]

    if val <= -1 and not in_drought:
        start = date
        in_drought = True

    elif val >= 0 and in_drought:
        end = date
        droughts.append((start, end))
        in_drought = False

# Save results
out_csv = os.path.join(analysis_folder, "drought_events_SPI3.csv")

pd.DataFrame(droughts, columns=["Start", "End"]).to_csv(out_csv, index=False)

print("✅ Drought events detected!")

### SPI1, SPI3, SPI6 PLOT for 2010

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ======================
# Folder
# ======================
folder = r"C:\Users\gerii\Documents\projects\tekeze\SPI"

spi_types = ["SPI1", "SPI3", "SPI6"]

months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

# SPI range
vmin = -2.5
vmax = 2.5

# ======================
# FIGURE (3 x 12)
# ======================
fig, axes = plt.subplots(3, 12, figsize=(22, 7))

# ======================
# LOOP SPI TYPES
# ======================
for r, spi in enumerate(spi_types):

    for c in range(12):

        ax = axes[r, c]

        month_str = f"{c+1:02d}"

        # find file
        file = f"{spi}_2010{month_str}.tif"
        path = os.path.join(folder, file)

        if not os.path.exists(path):
            ax.axis("off")
            continue

        with rasterio.open(path) as src:
            data = src.read(1).astype("float32")

            # SPI can be negative → keep full range
            data[data < -5] = np.nan

            bounds = src.bounds
            extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

        im = ax.imshow(
            data,
            extent=extent,
            origin="upper",
            cmap="BrBG",
            vmin=vmin,
            vmax=vmax
        )

        # ======================
        # TITLES
        # ======================
        if r == 0:
            ax.set_title(months[c], fontsize=8)

        if c == 0:
            ax.set_ylabel(spi, fontsize=12)

        # ======================
        # GRID (1 degree)
        # ======================
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.yaxis.set_major_locator(MultipleLocator(1))
        ax.tick_params(labelsize=8)

        # reduce clutter
        if r < 2:
            ax.set_xticklabels([])
        if c > 0:
            ax.set_yticklabels([])

# ======================
# COLORBAR
# ======================
cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("SPI", fontsize=14)

# ======================
# TITLE
# ======================
plt.suptitle("Monthly SPI (1, 3, 6) – 2010", fontsize=14)

plt.subplots_adjust(wspace=0.02, hspace=0.02)

plt.show()

In [ ]:
# ======================
# SAVE FIGURE
# ======================
out_path = r"C:\Users\gerii\Documents\projects\tekeze\SPI\SPI_2010_3x12.png"

fig.canvas.draw()

fig.savefig(
    out_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("Saved figure to:", out_path)

### SPI12

In [ ]:
import os
import numpy as np
import rasterio
from scipy.stats import gamma, norm

# ============================================================
# 1️⃣ LOAD DATA (1990–2025)
# ============================================================
folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS monthly"

files = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])

data_list = []
dates = []

print("Loading rainfall data...")

for f in files:

    path = os.path.join(folder, f)

    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        meta = src.meta

    # mask negative values
    arr[arr < 0] = np.nan

    data_list.append(arr)

    # extract date from filename
    # example: 2000_01.tif
    dates.append(os.path.splitext(f)[0])

data = np.stack(data_list)

t, r, c = data.shape

print("Data shape:", data.shape)

# ============================================================
# 2️⃣ MOVING SUM
# ============================================================
def moving_sum(ts, k):

    out = np.full_like(ts, np.nan, dtype=float)

    for i in range(k - 1, len(ts)):

        window = ts[i - k + 1:i + 1]

        if np.isnan(window).any():
            continue

        out[i] = np.sum(window)

    return out

# ============================================================
# 3️⃣ SPI FUNCTION
# ============================================================
def compute_spi(ts, scale):

    # Step A: aggregated rainfall
    xk = moving_sum(ts, scale)

    valid = xk[~np.isnan(xk)]

    if len(valid) < 30:
        return np.full_like(ts, np.nan)

    # probability of zero rainfall
    q = np.sum(valid == 0) / len(valid)

    # fit gamma to non-zero values
    non_zero = valid[valid > 0]

    if len(non_zero) < 10:
        return np.full_like(ts, np.nan)

    try:
        alpha, loc, beta = gamma.fit(non_zero, floc=0)

    except:
        return np.full_like(ts, np.nan)

    spi = np.full_like(ts, np.nan, dtype=float)

    # compute SPI
    for i in range(len(xk)):

        if np.isnan(xk[i]):
            continue

        if xk[i] == 0:

            H = q

        else:

            G = gamma.cdf(xk[i], alpha, loc=loc, scale=beta)

            H = q + (1 - q) * G

        # avoid infinities
        H = np.clip(H, 1e-6, 1 - 1e-6)

        spi[i] = norm.ppf(H)

    return spi

# ============================================================
# 4️⃣ COMPUTE SPI-12
# ============================================================
print("\nComputing SPI-12...")

SPI12 = np.full((t, r, c), np.nan, dtype=np.float32)

for i in range(r):

    print(f"Processing row {i+1}/{r}")

    for j in range(c):

        ts = data[:, i, j]

        if np.all(np.isnan(ts)):
            continue

        SPI12[:, i, j] = compute_spi(ts, 12)

print("\n✅ SPI-12 computed successfully")

# ============================================================
# 5️⃣ SAVE OUTPUTS
# ============================================================
out_folder = os.path.join(folder, "SPI_12")
os.makedirs(out_folder, exist_ok=True)

meta.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan
)

print("\nSaving SPI-12 GeoTIFFs...")

for k in range(t):

    # output filename similar to SPEI
    out_name = f"SPI12_{dates[k]}.tif"

    out_path = os.path.join(out_folder, out_name)

    with rasterio.open(out_path, 'w', **meta) as dst:

        dst.write(SPI12[k].astype(np.float32), 1)

print("\n✅ All SPI-12 maps saved successfully!")

### Barplot

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd

# =====================================================
# MAIN FOLDER
# =====================================================
base_folder = r"C:\Users\gerii\Documents\projects\tekeze\indices\SPI"

scales = [1, 3, 6, 12]

# =====================================================
# FUNCTION TO LOAD SPI SERIES
# =====================================================
def load_spi_series(folder, scale):

    # -------------------------------------------------
    # FILE PATTERN
    # -------------------------------------------------
    if scale == 12:

        files = sorted([
            f for f in os.listdir(folder)
            if f.startswith("SPI12_") and f.endswith(".tif")
        ])

    else:

        files = sorted([
            f for f in os.listdir(folder)
            if f.startswith(f"SPI{scale}_") and f.endswith(".tif")
        ])

    dates = []
    values = []

    # -------------------------------------------------
    # READ FILES
    # -------------------------------------------------
    for f in files:

        path = os.path.join(folder, f)

        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)

        # mask invalid values
        arr[arr < -9990] = np.nan

        # spatial mean
        mean_val = np.nanmean(arr)

        # -------------------------------------------------
        # EXTRACT DATE
        # -------------------------------------------------

        # SPI1_YYYYMM.tif
        # SPI3_YYYYMM.tif
        # SPI6_YYYYMM.tif
        # SPI12_CHIRPS_tekeze_YYYYMM.tif

        name = os.path.splitext(f)[0]

        if scale == 12:

            # last part = YYYYMM
            yyyymm = name.split("_")[-1]

        else:

            # SPI1_YYYYMM
            yyyymm = name.split("_")[1]

        year = yyyymm[:4]
        month = yyyymm[4:6]

        date = pd.to_datetime(f"{year}-{month}")

        dates.append(date)
        values.append(mean_val)

    return dates, values

# =====================================================
# LOAD ALL SPI DATA
# =====================================================
all_data = {}

for scale in scales:

    print(f"Loading SPI-{scale}...")

    d, v = load_spi_series(base_folder, scale)

    all_data[scale] = (d, v)

# =====================================================
# CREATE FIGURE
# =====================================================
fig, axes = plt.subplots(
    4,
    1,
    figsize=(20, 14),
    sharex=True
)

# =====================================================
# PLOT EACH SCALE
# =====================================================
for ax, scale in zip(axes, scales):

    dates, values = all_data[scale]

    # positive = blue
    # negative = red
    colors = [
        "blue" if v >= 0 else "red"
        for v in values
    ]

    ax.bar(
        dates,
        values,
        color=colors,
        width=25
    )

    # zero line
    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # subplot title
    ax.set_title(
        f"SPI-{scale}",
        fontsize=25,   
    )
#fontweight='bold'
    # y label
    ax.set_ylabel(
        "SPI",
        fontsize=25
    )

    # larger tick labels
    ax.tick_params(
        axis='both',
        labelsize=25
    )

    # grid
    ax.grid(
        True,
        linestyle='--',
        alpha=0.4
    )

# =====================================================
# X LABEL
# =====================================================
plt.xlabel(
    "Year",
    fontsize=25
)

# =====================================================
# LAYOUT
# =====================================================
plt.tight_layout()

# =====================================================
# SAVE FIGURE
# =====================================================
out_path = os.path.join(
    base_folder,
    "SPI_4panel_barplot22.png"
)

plt.savefig(
    out_path,
    dpi=350,
    bbox_inches='tight'
)

print(f"\nSaved figure:\n{out_path}")

# =====================================================
# SHOW FIGURE
# =====================================================
plt.show()